# CSIT 595 Assignment 1: Unconditional Fashion-MNIST generative models

This notebook trains and compares a VAE, a GAN, and a DDPM-style diffusion model on Fashion-MNIST.
The experiment is unconditional: labels are only used by the evaluator after generation.

Set `CONFIG['full_run'] = False` for a quick smoke test or `True` for the final full experiment.

In [ ]:

import math
import random
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm.auto import tqdm

CONFIG = dict(
    full_run=True,
    seed=559,
    batch_size=128,
    eval_epochs=1,
    vae_epochs=1,
    gan_epochs=1,
    diffusion_steps=500,
    diffusion_train_steps=500,
    metric_count=500,
    sample_count=100,
    large_sample_count=5000,
    num_workers=0,
)
if CONFIG["full_run"]:
    CONFIG.update(
        eval_epochs=5,
        vae_epochs=20,
        gan_epochs=20,
        diffusion_train_steps=30000,
        metric_count=5000,
        sample_count=100,
        large_sample_count=5000,
        num_workers=0,
    )

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT = Path.cwd()
OUT = ROOT / "assignment1_outputs"
OUT.mkdir(exist_ok=True)

evaluator = None
vae = None
generator = None
discriminator = None
denoiser = None

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])

transform = transforms.Compose([
    transforms.Pad(2),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

train_set = datasets.FashionMNIST(ROOT / "data", train=True, download=True, transform=transform)
test_set = datasets.FashionMNIST(ROOT / "data", train=False, download=True, transform=transform)
train_loader = DataLoader(
    train_set,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    drop_last=True,
    num_workers=CONFIG["num_workers"],
)
test_loader = DataLoader(
    test_set,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(CONFIG["seed"])


class EvaluatorCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x, return_features=False):
        features = self.features(x)
        logits = self.classifier(features)
        if return_features:
            return logits, features.squeeze(-1).squeeze(-1)
        return logits


class VAE(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, 4, stride=2, padding=1),
            nn.ReLU(),
        )
        self.fc_mu = nn.Linear(128 * 2 * 2, latent_dim)
        self.fc_logvar = nn.Linear(128 * 2 * 2, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, 128 * 2 * 2)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 4, stride=2, padding=1),
            nn.Tanh(),
        )

    def encode(self, x):
        h = self.encoder(x)
        h = h.view(h.size(0), -1)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.fc_decode(z)
        h = h.view(-1, 128, 2, 2)
        x = self.decoder(h)
        return x

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar


class Generator(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.latent_dim = latent_dim
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 256, 4, 1, 0, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 1, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        z = z.view(z.size(0), z.size(1), 1, 1)
        return self.net(z)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 1, 4, 1, 0),
        )

    def forward(self, x):
        return self.net(x).view(x.size(0), -1)


class TinyUNet(nn.Module):
    def __init__(self, base_channels=32, time_dim=64):
        super().__init__()
        self.time_dim = time_dim
        self.time_proj = nn.Sequential(
            nn.Linear(time_dim, 128),
            nn.SiLU(),
            nn.Linear(128, 128),
        )
        self.inc = nn.Conv2d(1, base_channels, 3, padding=1)
        self.down1 = nn.Sequential(
            nn.Conv2d(base_channels, base_channels * 2, 4, stride=2, padding=1),
            nn.GroupNorm(8, base_channels * 2),
            nn.SiLU(),
        )
        self.down2 = nn.Sequential(
            nn.Conv2d(base_channels * 2, base_channels * 4, 4, stride=2, padding=1),
            nn.GroupNorm(8, base_channels * 4),
            nn.SiLU(),
        )
        self.mid = nn.Sequential(
            nn.Conv2d(base_channels * 4, base_channels * 4, 3, padding=1),
            nn.GroupNorm(8, base_channels * 4),
            nn.SiLU(),
            nn.Conv2d(base_channels * 4, base_channels * 4, 3, padding=1),
            nn.GroupNorm(8, base_channels * 4),
            nn.SiLU(),
        )
        self.up1 = nn.Sequential(
            nn.ConvTranspose2d(base_channels * 4, base_channels * 2, 4, stride=2, padding=1),
            nn.GroupNorm(8, base_channels * 2),
            nn.SiLU(),
        )
        self.up0 = nn.Sequential(
            nn.ConvTranspose2d(base_channels * 2, base_channels, 4, stride=2, padding=1),
            nn.GroupNorm(8, base_channels),
            nn.SiLU(),
        )
        self.out = nn.Conv2d(base_channels, 1, 3, padding=1)

    def forward(self, x, timestep):
        t = torch.tensor([timestep], device=x.device, dtype=torch.float32).repeat(x.size(0))
        t = torch.sin(t / 1000.0)[:, None]
        cond = self.time_proj(t)
        x0 = self.inc(x)
        x1 = self.down1(x0)
        x2 = self.down2(x1)
        m = self.mid(x2)
        u1 = self.up1(m)
        u1 = u1 + x1
        u0 = self.up0(u1)
        u0 = u0 + x0
        h = self.out(u0)
        return h


class Diffusion:
    def __init__(self, steps=500):
        self.steps = steps
        self.betas = torch.linspace(1e-4, 0.02, steps, device=DEVICE)
        self.alphas = 1.0 - self.betas
        self.alpha_bars = torch.cumprod(self.alphas, dim=0)

    def add_noise(self, clean, t, noise=None):
        if noise is None:
            noise = torch.randn_like(clean)
        alpha_bar = self.alpha_bars[t].to(clean.device).view(-1, 1, 1, 1)
        return torch.sqrt(alpha_bar) * clean + torch.sqrt(1.0 - alpha_bar) * noise

    def sample(self, denoiser, count, steps=None):
        if steps is None:
            steps = self.steps
        denoiser.eval()
        x = torch.randn(count, 1, 32, 32, device=DEVICE)
        for step in reversed(range(steps)):
            t = torch.full((count,), step, device=DEVICE, dtype=torch.long)
            with torch.no_grad():
                pred_noise = denoiser(x, t)
            if step > 0:
                beta = self.betas[step - 1].to(DEVICE)
                alpha = self.alphas[step - 1].to(DEVICE)
                alpha_bar = self.alpha_bars[step - 1].to(DEVICE)
                x = x - (beta / torch.sqrt(1.0 - alpha_bar)) * pred_noise
                x = x / torch.sqrt(alpha)
        return x.clamp(-1, 1)


def show_grid(images, title, filename, columns=10):
    images = ((images.detach().cpu().clamp(-1, 1) + 1) / 2).numpy()
    rows = max(1, math.ceil(len(images) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(columns, rows))
    axes = np.asarray(axes).reshape(-1)
    for idx, ax in enumerate(axes):
        ax.axis("off")
        if idx < len(images):
            ax.imshow(images[idx, 0], cmap="gray", vmin=0, vmax=1)
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(OUT / filename, dpi=160)
    plt.close(fig)


@torch.no_grad()
def evaluator_outputs(images):
    logits, features = evaluator(images.to(DEVICE), return_features=True)
    return logits.argmax(1).cpu(), features.cpu()


def evaluator_accuracy():
    evaluator.eval()
    correct = 0
    total = 0
    for images, labels in test_loader:
        logits = evaluator(images.to(DEVICE))
        correct += (logits.argmax(1).cpu() == labels).sum().item()
        total += len(labels)
    return correct / total


def sqrt_psd(matrix):
    values, vectors = np.linalg.eigh((matrix + matrix.T) / 2)
    values = np.clip(values, 0, None)
    return (vectors * np.sqrt(values)) @ vectors.T


def coverage_metrics(real_features, real_labels, generated_features, generated_labels):
    real_mean = real_features.mean(0)
    generated_mean = generated_features.mean(0)
    real_cov = np.cov(real_features, rowvar=False) + 1e-6 * np.eye(real_features.shape[1])
    generated_cov = np.cov(generated_features, rowvar=False) + 1e-6 * np.eye(generated_features.shape[1])
    cfid = np.sum((real_mean - generated_mean) ** 2) + np.trace(real_cov + generated_cov - 2 * sqrt_psd(real_cov @ generated_cov))

    real_hist = np.bincount(real_labels, minlength=10).astype(np.float64)
    real_hist /= real_hist.sum()
    gen_hist = np.bincount(generated_labels, minlength=10).astype(np.float64)
    gen_hist /= gen_hist.sum()
    entropy = -np.sum(gen_hist * np.log(gen_hist + 1e-8))
    kl = np.sum(gen_hist * np.log((gen_hist + 1e-8) / (real_hist + 1e-8)))
    return {
        "cfid": float(cfid),
        "entropy": float(entropy),
        "kl_divergence": float(kl),
        "generated_histogram": gen_hist.tolist(),
    }


def reference_features(count):
    all_features = []
    all_labels = []
    for images, labels in test_loader:
        if sum(len(x) for x in all_labels) >= count:
            break
        remaining = count - sum(len(x) for x in all_labels)
        images = images[:remaining]
        labels = labels[:remaining]
        _, features = evaluator_outputs(images)
        all_features.append(features)
        all_labels.append(labels)
    return torch.cat(all_features).numpy(), torch.cat(all_labels).numpy()


def predicted_histogram(images):
    labels = []
    for start in range(0, len(images), CONFIG["batch_size"]):
        batch_labels, _ = evaluator_outputs(images[start : start + CONFIG["batch_size"]])
        labels.append(batch_labels)
    hist = torch.bincount(torch.cat(labels), minlength=10).float()
    return hist / hist.sum()


def train_evaluator(model):
    optimizer = torch.optim.Adam(model.parameters(), 2e-3)
    history = []
    model.train()
    for epoch in range(CONFIG["eval_epochs"]):
        running_loss = 0.0
        correct = 0
        total = 0
        for images, labels in tqdm(train_loader, desc=f"Evaluator {epoch + 1}", leave=False):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = F.cross_entropy(logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)
        history.append({"epoch": epoch + 1, "loss": running_loss / total, "acc": correct / total})
    return model, history


def train_vae(model):
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)
    history = []
    started = time.perf_counter()
    for epoch in range(CONFIG["vae_epochs"]):
        model.train()
        total_recon = 0.0
        total_kl = 0.0
        total_loss = 0.0
        for images, _ in tqdm(train_loader, desc=f"VAE {epoch + 1}", leave=False):
            images = images.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            recon, mu, logvar = model(images)
            recon_loss = F.mse_loss(recon, images, reduction="mean")
            kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
            loss = recon_loss + 0.1 * kl
            loss.backward()
            optimizer.step()
            total_recon += recon_loss.item()
            total_kl += kl.item()
            total_loss += loss.item()
        history.append({
            "epoch": epoch + 1,
            "recon_loss": total_recon / len(train_loader),
            "kl": total_kl / len(train_loader),
            "total_loss": total_loss / len(train_loader),
        })
    return model, history, time.perf_counter() - started


@torch.no_grad()
def sample_vae(count):
    z = torch.randn(count, vae.latent_dim, device=DEVICE)
    return vae.decode(z).clamp(-1, 1)


def train_gan(generator, discriminator):
    g_opt = torch.optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
    d_opt = torch.optim.Adam(discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))
    history = []
    started = time.perf_counter()
    for epoch in range(CONFIG["gan_epochs"]):
        generator.train(); discriminator.train()
        g_loss_total = 0.0
        d_loss_total = 0.0
        for images, _ in tqdm(train_loader, desc=f"GAN {epoch + 1}", leave=False):
            images = images.to(DEVICE)
            batch = images.size(0)
            real_label = torch.ones(batch, device=DEVICE)
            fake_label = torch.zeros(batch, device=DEVICE)

            d_opt.zero_grad(set_to_none=True)
            real_logits = discriminator(images)
            real_loss = F.binary_cross_entropy_with_logits(real_logits.view(-1), real_label)
            noise = torch.randn(batch, generator.latent_dim, device=DEVICE)
            fake_images = generator(noise)
            fake_logits = discriminator(fake_images.detach())
            fake_loss = F.binary_cross_entropy_with_logits(fake_logits.view(-1), fake_label)
            d_loss = real_loss + fake_loss
            d_loss.backward()
            d_opt.step()

            g_opt.zero_grad(set_to_none=True)
            noise = torch.randn(batch, generator.latent_dim, device=DEVICE)
            fake_images = generator(noise)
            fake_logits = discriminator(fake_images)
            g_loss = F.binary_cross_entropy_with_logits(fake_logits.view(-1), real_label)
            g_loss.backward()
            g_opt.step()

            g_loss_total += g_loss.item()
            d_loss_total += d_loss.item()
        history.append({"epoch": epoch + 1, "g_loss": g_loss_total / len(train_loader), "d_loss": d_loss_total / len(train_loader)})
    return generator, discriminator, history, time.perf_counter() - started


@torch.no_grad()
def sample_gan(count):
    noise = torch.randn(count, generator.latent_dim, device=DEVICE)
    return generator(noise).clamp(-1, 1)


def train_diffusion(denoiser):
    optimizer = torch.optim.AdamW(denoiser.parameters(), lr=1e-4)
    history = []
    diffusion = Diffusion(steps=CONFIG["diffusion_steps"])
    started = time.perf_counter()
    denoiser.train()
    for step in range(CONFIG["diffusion_train_steps"]):
        images, _ = next(iter(train_loader))
        batch = images.to(DEVICE)
        t = torch.randint(0, diffusion.steps, (batch.size(0),), device=DEVICE)
        noise = torch.randn_like(batch)
        noisy = diffusion.add_noise(batch, t, noise=noise)
        pred = denoiser(noisy, t)
        loss = F.mse_loss(pred, noise)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        if step % max(1, CONFIG["diffusion_train_steps"] // 10) == 0:
            history.append({"step": step + 1, "mse": loss.item()})
    return denoiser, diffusion, history, time.perf_counter() - started


def evaluate_model(name, sample_fn, training_seconds=None, parameters=None):
    count = CONFIG["metric_count"]
    started = time.perf_counter()
    generated = sample_fn(count)
    elapsed = time.perf_counter() - started
    labels, features = evaluator_outputs(generated)
    real_features, real_labels = reference_features(count)
    metrics = coverage_metrics(real_features, real_labels, features.numpy(), labels.numpy())
    metrics.update(
        model=name,
        images_per_second=count / elapsed,
        time_for_5000_images=5000 * elapsed / count,
        parameters=parameters,
        training_seconds=training_seconds,
    )
    return metrics, generated


def run_experiment():
    global evaluator, vae, generator, discriminator, denoiser
    print("Training evaluator...")
    evaluator = EvaluatorCNN().to(DEVICE)
    evaluator, eval_hist = train_evaluator(evaluator)
    print("Evaluator test accuracy:", evaluator_accuracy())

    print("Training VAE...")
    vae = VAE(latent_dim=64).to(DEVICE)
    vae, vae_hist, vae_training_seconds = train_vae(vae)

    print("Training GAN...")
    generator = Generator(latent_dim=128).to(DEVICE)
    discriminator = Discriminator().to(DEVICE)
    generator, discriminator, gan_hist, gan_training_seconds = train_gan(generator, discriminator)

    print("Training diffusion model...")
    denoiser = TinyUNet(base_channels=32, time_dim=64).to(DEVICE)
    denoiser, diffusion, diffusion_hist, diffusion_training_seconds = train_diffusion(denoiser)

    vae_metrics, _ = evaluate_model(
        "VAE",
        sample_vae,
        training_seconds=vae_training_seconds,
        parameters=sum(p.numel() for p in vae.parameters()),
    )
    gan_metrics, _ = evaluate_model(
        "GAN",
        sample_gan,
        training_seconds=gan_training_seconds,
        parameters=sum(p.numel() for p in generator.parameters()) + sum(p.numel() for p in discriminator.parameters()),
    )
    ddpm_metrics, _ = evaluate_model(
        "DDPM",
        lambda count: diffusion.sample(denoiser, count, steps=CONFIG["diffusion_steps"]),
        training_seconds=diffusion_training_seconds,
        parameters=sum(p.numel() for p in denoiser.parameters()),
    )

    results = pd.DataFrame([vae_metrics, gan_metrics, ddpm_metrics])
    results.to_csv(OUT / "results.csv", index=False)
    print(results)

    preview_images, _ = next(iter(test_loader))
    preview_images = preview_images[:10].to(DEVICE)
    with torch.no_grad():
        recon, _, _ = vae(preview_images)
        z0 = torch.randn(1, vae.latent_dim, device=DEVICE)
        z1 = torch.randn(1, vae.latent_dim, device=DEVICE)
        interpolated = []
        for frac in torch.linspace(0.0, 1.0, 8):
            z = torch.lerp(z0, z1, frac.item())
            interpolated.append(vae.decode(z)[0].unsqueeze(0))
        vae_interp = torch.cat(interpolated, dim=0)

    show_grid(torch.cat([preview_images.cpu(), recon.cpu()]), "VAE originals (top) and reconstructions (bottom)", "vae_reconstruction.png")
    show_grid(vae_interp.cpu(), "VAE latent interpolation", "vae_interpolation.png", columns=8)

    with torch.no_grad():
        gan_interp = []
        z0 = torch.randn(1, generator.latent_dim, device=DEVICE)
        z1 = torch.randn(1, generator.latent_dim, device=DEVICE)
        for frac in torch.linspace(0.0, 1.0, 8):
            z = torch.lerp(z0, z1, frac.item())
            gan_interp.append(generator(z)[0].unsqueeze(0))
        gan_interp = torch.cat(gan_interp, dim=0).cpu()
    show_grid(gan_interp, "GAN latent interpolation", "gan_interpolation.png", columns=8)

    gan_hist = predicted_histogram(sample_gan(CONFIG["sample_count"]))
    plt.figure(figsize=(7, 3))
    plt.bar(range(10), gan_hist.numpy())
    plt.xticks(range(10))
    plt.title("GAN mode-collapse watch: evaluator-predicted label histogram")
    plt.tight_layout()
    plt.savefig(OUT / "gan_mode_collapse_watch.png", dpi=160)
    plt.close()

    diff_rows = []
    for steps in [25, 100, CONFIG["diffusion_steps"]]:
        t0 = time.perf_counter()
        x = diffusion.sample(denoiser, CONFIG["metric_count"], steps=steps)
        elapsed = time.perf_counter() - t0
        labels, features = evaluator_outputs(x)
        ref_features, ref_labels = reference_features(CONFIG["metric_count"])
        metrics = coverage_metrics(ref_features, ref_labels, features.numpy(), labels.numpy())
        metrics["sampling_steps"] = steps
        metrics["images_per_second"] = CONFIG["metric_count"] / elapsed
        metrics["time_for_5000_images"] = 5000 * elapsed / CONFIG["metric_count"]
        diff_rows.append(metrics)
    pd.DataFrame(diff_rows).to_csv(OUT / "diffusion_speed_quality.csv", index=False)

    show_grid(sample_vae(CONFIG["sample_count"]), "VAE prior samples", "vae_samples.png")
    show_grid(sample_gan(CONFIG["sample_count"]), "GAN samples", "gan_samples.png")
    show_grid(diffusion.sample(denoiser, CONFIG["sample_count"], steps=min(CONFIG["diffusion_steps"], 100)), "DDPM samples", "diffusion_samples.png")

    print("Saved artifacts to:", OUT)


if __name__ == "__main__":
    run_experiment()
